In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn import svm as svm
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import confusion_matrix as cm
from sklearn.model_selection import StratifiedKFold

## Loading data

In [129]:
def _to_array(raw_features):
    """
    Convert stored feature payload to a 3D array: (windows, channels, n_features).
    Handles the dict stored in features.npz.
    """
    arr = np.asarray(raw_features)
    if arr.dtype == object and arr.shape == ():  # dict wrapped in 0-d object
        obj = arr.item()
        if isinstance(obj, dict):
            keys = sorted(obj.keys())  # consistent order
            stacked = np.stack([np.asarray(obj[k]) for k in keys], axis=-1)
            return stacked, keys
        return np.asarray(obj), None
    return arr, None

In [130]:
def load_feature_files(root="data", pattern="*_features.npz", target_channels=None):
      """Load features and standardize channel count by padding/truncating.
      If target_channels is None, use the max channel count found (pads smaller files)."""
      root = Path(root)
      collected = []
      for fp in sorted(root.rglob(pattern)):
          data = np.load(fp, allow_pickle=True)
          X_raw = data["features"]
          X, feat_keys = _to_array(X_raw)
          labels = data.get("window_labels")
          labels = np.array([None] * len(X), dtype=object) if labels is None else np.asarray(labels, dtype=object)
          collected.append((fp, X, labels, feat_keys, data))
      if not collected:
          raise FileNotFoundError(f"No feature files found under {root}")

      if target_channels is None:
          target_channels = max(item[1].shape[1] for item in collected)

      X_list, y_list, meta = [], [], []
      for fp, X, labels, feat_keys, data in collected:
          channels = X.shape[1]
          if channels < target_channels:
              pad = np.zeros((X.shape[0], target_channels - channels, X.shape[2]), dtype=X.dtype)
              X = np.concatenate([X, pad], axis=1)
          elif channels > target_channels:
              X = X[:, :target_channels, :]
          X_list.append(X)
          y_list.append(labels)
          meta.append({
              "path": str(fp),
              "feature_list": feat_keys or list(data["feature_list"]),
              "fs": float(np.asarray(data["fs"]).squeeze()),
              "window_size_samples": int(data["window_size_samples"]),
              "window_step_samples": int(data["window_step_samples"]),
              "window_start_samples": np.asarray(data["window_start_samples"]),
              "window_end_samples": np.asarray(data["window_end_samples"]),
              "window_label_confidence": data.get("window_label_confidence"),
          })
      return np.vstack(X_list), np.concatenate(y_list), meta

In [131]:
def load_features_by_label(root="data", pattern="*_features.npz", target_channels=None):
      X, y, meta = load_feature_files(root, pattern, target_channels=target_channels)
      mask = y != None  # noqa: E711
      return X[mask], y[mask], meta

### Classifiers, Lda

In [ ]:
# Load and split data
X_raw, y, meta = load_features_by_label(target_channels=None)  # auto-pads to max channel count
X = X_raw.reshape(X_raw.shape[0], -1)  # flatten channels x features

X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, stratify=y, random_state=42
  )

# Eval helper: 10-fold CV + holdout


def eval_model(name, model, X, y):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_scores = cross_validate(model, X, y, cv=cv, scoring='accuracy', return_train_score=False)
    print(f"{name} 10-fold CV accuracy: {cv_scores['test_score'].mean():.3f} +/- {cv_scores['test_score'].std():.3f}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"{name} holdout accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print('Confusion matrix:\n', cm(y_test, y_pred))
    print('\nReport:\n', classification_report(y_test, y_pred))

Test accuracy: 0.774500475737393
              precision    recall  f1-score   support

   left_turn       0.88      0.77      0.82       429
     neutral       0.56      0.80      0.66       233
  right_turn       0.87      0.77      0.82       389

    accuracy                           0.77      1051
   macro avg       0.77      0.78      0.76      1051
weighted avg       0.81      0.77      0.78      1051



In [ ]:
# LDA
lda = LDA(store_covariance=True)
eval_model('LDA', lda, X, y)


In [ ]:
# QDA
qda = QDA()
eval_model('QDA', qda, X, y)

In [ ]:
# SVM (scale features)
svm_clf = make_pipeline(StandardScaler(), svm.SVC(kernel='rbf', C=100, gamma='scale'))
eval_model('SVM (RBF)', svm_clf, X, y)

In [ ]:
# Random Forest
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1)
eval_model('Random Forest', rf_clf, X, y)